# Notebook 7 — Filter Methods for Feature Selection

**Difficulty:** ⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 55 min

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Implement a **Variance Threshold** filter from scratch and verify it against scikit-learn.
2. Implement a **Pearson Correlation** filter from scratch and understand its limitations.
3. Implement **Mutual Information** from scratch and explain why it catches non-linear relationships.
4. Implement a **Chi-square** test from scratch for categorical features.
5. Build a **redundancy-removal** step that drops highly correlated feature pairs.
6. Chain these steps into a full **filter pipeline** and measure the impact on model performance.

**Dataset theme:** Airbnb-style listing price prediction (numeric, categorical, and derived features).


## The Analogy: Resume Screening vs. Job Interviews

Imagine a hiring manager receives 200 applications for 5 open roles.

**Filter methods** are like the *initial resume screen* — no interview needed.
The manager skims each resume **independently** and scores it on a few quick signals:
years of experience, relevant keywords, GPA.
This screen is **fast and cheap** — it takes seconds per resume — but it can miss candidates
whose strengths only show up in combination (e.g., someone with modest experience who
happens to have the *exact* skill your team is missing).

**Wrapper methods** (next notebook) are like actually running candidates through interviews
with your real team. Thorough, but expensive: every candidate costs hours of engineering time.

**Key insight:** Filter methods score features **independently of any model**.
They are purely statistical checks: variance, correlation, mutual information, chi-square.
That makes them blazing fast — perfect for a first-pass on high-dimensional data.


## Why Does This Matter for ML?

In real ML projects, raw feature tables often contain **hundreds to thousands of columns**
after one-hot encoding and feature engineering.
Training a model on all of them is slow, prone to overfitting, and hard to interpret.

Filter methods run in **O(n × p)** time — linear in both samples and features —
making them the only practical first-pass when p > 500.

A typical production workflow:

```
Raw features (p=1000)
    → Variance threshold   → ~800 features
    → MI / Pearson ranking → ~100 features
    → Redundancy removal   → ~50 features
    → Wrapper / embedded   → ~10–20 final features
```

Filter methods do the heavy lifting so that wrapper methods (which are O(p²)) are tractable.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, mutual_info_regression, chi2
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import KBinsDiscretizer
from scipy import stats

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Generate synthetic Airbnb-style dataset ────────────────────────────────
N = 1_000   # number of listings

# ── 5 truly predictive features ───────────────────────────────────────────
bedrooms        = np.random.randint(1, 6, N).astype(float)          # 1–5 bedrooms
bathrooms       = np.random.randint(1, 4, N).astype(float)          # 1–3 baths
dist_downtown   = np.random.exponential(3, N)                        # km to city centre
review_score    = np.random.uniform(3.5, 5.0, N)                    # 3.5–5.0 stars
amenities_count = np.random.poisson(12, N).astype(float)            # number of amenities

# ── 5 redundant features (linearly correlated with predictive ones) ────────
total_rooms   = bedrooms + bathrooms + np.random.normal(0, 0.2, N)  # ≈ bedrooms+bathrooms
dist_noisy    = dist_downtown + np.random.normal(0, 0.5, N)         # noisy copy of dist
score_noisy   = review_score + np.random.normal(0, 0.05, N)         # noisy copy of score
bed_sq        = bedrooms ** 2 + np.random.normal(0, 0.1, N)         # correlated with bedrooms
amenity_norm  = (amenities_count - amenities_count.mean()) / amenities_count.std()  # scaled copy

# ── 5 pure noise features ─────────────────────────────────────────────────
noise1 = np.random.normal(0, 1, N)
noise2 = np.random.uniform(0, 1, N)
noise3 = np.random.normal(5, 2, N)
noise4 = np.random.exponential(1, N)
noise5 = np.random.randint(0, 100, N).astype(float)

# ── Combine into matrix ───────────────────────────────────────────────────
feature_names = [
    'bedrooms', 'bathrooms', 'dist_downtown', 'review_score', 'amenities_count',  # predictive
    'total_rooms', 'dist_noisy', 'score_noisy', 'bed_sq', 'amenity_norm',          # redundant
    'noise1', 'noise2', 'noise3', 'noise4', 'noise5'                               # pure noise
]
X = np.column_stack([
    bedrooms, bathrooms, dist_downtown, review_score, amenities_count,
    total_rooms, dist_noisy, score_noisy, bed_sq, amenity_norm,
    noise1, noise2, noise3, noise4, noise5
])

# ── Target: log(price) driven by predictive features ─────────────────────
log_price = (
    0.4  * bedrooms
    + 0.3  * bathrooms
    - 0.15 * dist_downtown
    + 0.5  * review_score
    + 0.1  * amenities_count
    + np.random.normal(0, 0.3, N)   # irreducible noise
)
y = log_price

print(f"Dataset shape: {X.shape}")
print(f"Feature groups: 5 predictive | 5 redundant | 5 noise")
print(f"Target (log price): mean={y.mean():.2f}, std={y.std():.2f}")


## Three Categories of Filter Methods

| Category | Methods | What it measures | Best for |
|---|---|---|---|
| **Univariate statistics** | Pearson r, Mutual Information, Chi-square | Feature–target association | Initial ranking |
| **Redundancy filter** | Pairwise correlation matrix | Feature–feature correlation | After ranking |
| **Variance threshold** | Var(feature) | Near-constant features | Very first step |

We apply them in this order: **variance → ranking → redundancy removal**.


## Step 1 — Variance Threshold

### Intuition

A feature with **zero (or near-zero) variance** is essentially a constant — it takes the
same value for every listing. A constant column can never help a model discriminate between
high-price and low-price listings, so we drop it immediately.

**Formula:** $\text{Var}(X_j) = \frac{1}{n} \sum_{i=1}^{n} (x_{ij} - \bar{x}_j)^2$

Rule of thumb: drop any feature whose variance is below a small threshold (e.g. 0.01).
For binary features the threshold is often set to $p(1-p)$ where $p$ is the proportion of 1s.


In [ ]:
np.random.seed(42)

# Add 3 near-constant features to expose the variance threshold
near_const1 = np.ones(N) * 5.0 + np.random.normal(0, 0.0001, N)   # almost always 5
near_const2 = np.zeros(N)                                            # truly constant
near_const3 = np.where(np.random.rand(N) < 0.001, 1.0, 0.0)        # 99.9% zeros

X_aug = np.column_stack([X, near_const1, near_const2, near_const3])
aug_names = feature_names + ['near_const1', 'near_const2', 'near_const3']

# ── From-scratch implementation ───────────────────────────────────────────
def variance_threshold_scratch(X, threshold=0.01):
    """Return boolean mask of features whose variance exceeds `threshold`."""
    variances = X.var(axis=0)          # per-column variance
    selected  = variances > threshold  # True = keep
    return selected, variances

mask_scratch, variances = variance_threshold_scratch(X_aug, threshold=0.01)

print("=== Variance Threshold (scratch) ===")
for name, var, keep in zip(aug_names, variances, mask_scratch):
    status = 'KEEP' if keep else 'DROP'
    print(f"  {name:<20}  var={var:.6f}  → {status}")

# ── Verify against sklearn ────────────────────────────────────────────────
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_aug)
mask_sklearn = vt.get_support()

match = np.all(mask_scratch == mask_sklearn)
print(f"\nScratch == sklearn: {match}")
print(f"Features kept: {mask_scratch.sum()} / {len(aug_names)}")

# Work with the filtered matrix going forward
X_vt        = X_aug[:, mask_scratch]
names_vt    = [n for n, k in zip(aug_names, mask_scratch) if k]


## Step 2 — Pearson Correlation Filter

### Intuition

Pearson's r measures the **linear relationship** between a feature $X_j$ and the target $y$.
A value near +1 means "as $X_j$ goes up, $y$ goes up proportionally".
A value near 0 means "no linear trend" — but beware: r ≈ 0 does NOT mean no relationship!

**Formula:**
$r(X_j, y) = \frac{\sum_i (x_{ij} - \bar{x}_j)(y_i - \bar{y})}{\sqrt{\sum_i (x_{ij} - \bar{x}_j)^2 \cdot \sum_i (y_i - \bar{y})^2}}$

We rank features by $|r|$ and keep the top $k$.

**Pros:** Instantaneous to compute, easy to interpret.
**Cons:** Blind to non-linear associations.


In [ ]:
np.random.seed(42)

def pearson_filter(X, y, k=10):
    """Select top-k features by absolute Pearson correlation with y."""
    correlations = np.array([
        np.corrcoef(X[:, j], y)[0, 1]   # 2×2 matrix; [0,1] is the cross-correlation
        for j in range(X.shape[1])
    ])
    ranking = np.argsort(np.abs(correlations))[::-1]   # descending by |r|
    return ranking[:k], correlations

ranking_pearson, correlations = pearson_filter(X_vt, y, k=10)

print("=== Top features by |Pearson r| ===")
for rank, idx in enumerate(ranking_pearson):
    print(f"  {rank+1}. {names_vt[idx]:<20}  r = {correlations[idx]:+.4f}")

# ── Verify against sklearn SelectKBest(f_regression) ──────────────────────
# f_regression uses F-statistic which is a monotone transform of |r|, so rankings match
skb = SelectKBest(f_regression, k=10).fit(X_vt, y)
top_sklearn = np.argsort(skb.scores_)[::-1][:10]
overlap = len(set(ranking_pearson) & set(top_sklearn))
print(f"\nOverlap with sklearn SelectKBest(f_regression) top-10: {overlap}/10")

# ── Horizontal bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sorted_idx  = np.argsort(correlations)           # ascending for horizontal bar
sorted_corr = correlations[sorted_idx]
sorted_names = [names_vt[i] for i in sorted_idx]
colors = ['steelblue' if c >= 0 else 'tomato' for c in sorted_corr]
ax.barh(sorted_names, sorted_corr, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with log(price)')
ax.set_title('Feature–Target Pearson Correlation')
plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

# ── Limitation demo: U-shaped relationship ────────────────────────────────
# If y = x^2 + noise, x is perfectly predictive but Pearson r ≈ 0
x_demo = np.linspace(-3, 3, 500)
y_demo = x_demo ** 2 + np.random.normal(0, 0.3, 500)

r_demo = np.corrcoef(x_demo, y_demo)[0, 1]
print(f"Pearson r for y = x² + noise: {r_demo:.4f}  (should be near 0)")
print("Yet x is the only driver of y — Pearson completely misses this!")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x_demo, y_demo, alpha=0.4, s=10, color='steelblue')
ax.set_xlabel('x  (the feature)')
ax.set_ylabel('y = x² + noise')
ax.set_title(f'U-shaped relationship — Pearson r = {r_demo:.3f}')
plt.tight_layout()
plt.show()


## Step 3 — Mutual Information (MI)

### Intuition

**Mutual Information** asks: "How much does knowing $X_j$ reduce my uncertainty about $y$?"
Unlike Pearson r, it works for **any relationship** — linear, U-shaped, periodic — because
it is based on probability distributions, not linear trends.

To estimate MI for continuous variables we **discretize** them into $k$ equal-frequency bins,
build a joint frequency table, and apply the entropy formula:

$\text{MI}(X_j; Y) = \sum_{x}\sum_{y} p(x,y) \log\frac{p(x,y)}{p(x)\,p(y)}$

- If $X_j$ and $Y$ are **independent**, $p(x,y) = p(x)p(y)$ everywhere → $\text{MI} = 0$.
- If $X_j$ **fully determines** $Y$, the log term is always positive → $\text{MI} = H(Y)$ (maximum).

**Pros:** Model-free, detects any statistical dependence.
**Cons:** Slower than Pearson, sensitive to the number of bins.


In [ ]:
np.random.seed(42)

def mutual_information_scratch(x, y, n_bins=10):
    """
    Estimate MI between continuous vectors x and y.
    Both are discretized into n_bins equal-frequency (quantile) bins.
    """
    # Use percentile boundaries so each bin has roughly equal occupancy
    boundaries_x = np.percentile(x, np.linspace(0, 100, n_bins + 1)[1:-1])
    boundaries_y = np.percentile(y, np.linspace(0, 100, n_bins + 1)[1:-1])

    x_binned = np.digitize(x, boundaries_x)   # integer bin index 0..n_bins-1
    y_binned = np.digitize(y, boundaries_y)

    n = len(x)

    # Build joint frequency table (n_bins × n_bins)
    joint_counts = np.zeros((n_bins, n_bins))
    for xi, yi in zip(x_binned, y_binned):
        joint_counts[xi, yi] += 1          # tally each (bin_x, bin_y) pair

    pxy = joint_counts / n                 # joint probability table
    px  = pxy.sum(axis=1)                  # marginal over y  → shape (n_bins,)
    py  = pxy.sum(axis=0)                  # marginal over x  → shape (n_bins,)

    # MI = Σ p(x,y) log[ p(x,y) / (p(x)·p(y)) ]
    mi = 0.0
    for i in range(n_bins):
        for j in range(n_bins):
            if pxy[i, j] > 0 and px[i] > 0 and py[j] > 0:
                mi += pxy[i, j] * np.log(pxy[i, j] / (px[i] * py[j]))

    return max(0.0, mi)    # numerical noise can make MI slightly negative


# ── Demo: U-shaped feature ─────────────────────────────────────────────────
mi_ushaped  = mutual_information_scratch(x_demo, y_demo)
r_ushaped   = abs(np.corrcoef(x_demo, y_demo)[0, 1])
print(f"U-shaped feature:  |Pearson r| = {r_ushaped:.4f}  |  MI (scratch) = {mi_ushaped:.4f}")
print("MI correctly detects the non-linear relationship that Pearson misses.\n")

# ── Compute MI for all Airbnb features ────────────────────────────────────
mi_scores_scratch = np.array([
    mutual_information_scratch(X_vt[:, j], y) for j in range(X_vt.shape[1])
])

# ── Verify against sklearn mutual_info_regression ─────────────────────────
mi_sklearn = mutual_info_regression(X_vt, y, random_state=42)

print("=== MI scores: scratch vs. sklearn ===")
df_mi = pd.DataFrame({
    'feature':  names_vt,
    'MI_scratch': mi_scores_scratch,
    'MI_sklearn': mi_sklearn
}).sort_values('MI_sklearn', ascending=False)
print(df_mi.to_string(index=False))
print("\n(Scratch uses coarser discretization, so values differ but rankings are similar.)")


In [ ]:
np.random.seed(42)

# ── Head-to-head: Pearson rank vs. MI rank ─────────────────────────────────
abs_pearson = np.abs(correlations)

# Rank: 1 = best feature
pearson_rank = pd.Series(abs_pearson, index=names_vt).rank(ascending=False)
mi_rank      = pd.Series(mi_sklearn,  index=names_vt).rank(ascending=False)
rank_diff    = (pearson_rank - mi_rank).abs()

df_compare = pd.DataFrame({
    'Pearson_rank': pearson_rank.astype(int),
    'MI_rank':      mi_rank.astype(int),
    'rank_diff':    rank_diff.astype(int)
}).sort_values('rank_diff', ascending=False)

print("=== Features where Pearson and MI rank disagree most ===")
print(df_compare.to_string())
print("\nFeatures with large rank_diff have non-linear relationships with the target.")

# ── Bar chart comparing the two rankings ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sorted_p = pd.Series(abs_pearson, index=names_vt).sort_values()
axes[0].barh(sorted_p.index, sorted_p.values, color='steelblue')
axes[0].set_title('Ranking by |Pearson r|')
axes[0].set_xlabel('|Pearson r|')

sorted_m = pd.Series(mi_sklearn, index=names_vt).sort_values()
axes[1].barh(sorted_m.index, sorted_m.values, color='darkorange')
axes[1].set_title('Ranking by Mutual Information (sklearn)')
axes[1].set_xlabel('MI score')

plt.suptitle('Filter Rankings: Pearson vs. Mutual Information', fontsize=13)
plt.tight_layout()
plt.show()


## Step 4 — Chi-Square Test for Categorical Features

### Intuition

Pearson r and MI work for continuous (numeric) features.
For **categorical** features paired with a **categorical target** (e.g., is_expensive? yes/no),
we use the **chi-square ($\chi^2$) test of independence**.

The idea: build a contingency table of (feature category) × (target class).
If the feature is **independent** of the target, the observed counts should match the
*expected* counts calculated from the marginal totals.
A large $\chi^2$ statistic means the feature's distribution varies across target classes
— i.e., it carries information.

$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$

where $E_{ij} = \frac{(\text{row total}_i)(\text{col total}_j)}{\text{grand total}}$.


In [ ]:
np.random.seed(42)

# ── Create a binary classification version of Airbnb ─────────────────────
# is_expensive = 1 if log_price > median, else 0
y_binary = (y > np.median(y)).astype(int)

# ── Generate categorical features ─────────────────────────────────────────
room_type    = np.random.choice(['entire_apt', 'private_room', 'shared_room'], N,
                                 p=[0.55, 0.35, 0.10])   # informative: entire_apt costs more
neighborhood = np.random.choice(['downtown', 'midtown', 'suburbs', 'outskirts'], N,
                                 p=[0.25, 0.30, 0.30, 0.15])  # informative
host_language = np.random.choice(['english', 'spanish', 'other'], N,
                                   p=[0.60, 0.25, 0.15])  # noise: no real effect
has_parking   = np.random.choice(['yes', 'no'], N, p=[0.40, 0.60])  # mildly informative

# Encode to integers for chi2
from sklearn.preprocessing import OrdinalEncoder
cat_features_raw = np.column_stack([room_type, neighborhood, host_language, has_parking])
enc = OrdinalEncoder()
X_cat = enc.fit_transform(cat_features_raw).astype(int)
cat_names = ['room_type', 'neighborhood', 'host_language', 'has_parking']

# ── Chi-square from scratch ───────────────────────────────────────────────
def chi2_scratch(X_cat, y_cat):
    """Compute chi-squared statistic for each categorical feature vs. categorical target."""
    chi2_stats = []
    for j in range(X_cat.shape[1]):
        classes = np.unique(y_cat)                           # target class labels
        values  = np.unique(X_cat[:, j])                    # feature category labels
        # Build observed contingency table: rows=feature values, cols=target classes
        observed = np.array([
            [np.sum((X_cat[:, j] == v) & (y_cat == c)) for c in classes]
            for v in values
        ])
        # Expected counts from marginal totals under the independence assumption
        row_sums = observed.sum(axis=1, keepdims=True)      # per-feature-category totals
        col_sums = observed.sum(axis=0, keepdims=True)      # per-target-class totals
        expected = row_sums * col_sums / observed.sum()     # outer product / N
        # Chi-squared statistic (add 1e-10 to avoid division by zero)
        chi2_val = np.sum((observed - expected) ** 2 / (expected + 1e-10))
        chi2_stats.append(chi2_val)
    return np.array(chi2_stats)

chi2_scores_scratch = chi2_scratch(X_cat, y_binary)

# ── Verify against sklearn chi2 ───────────────────────────────────────────
chi2_scores_sklearn, p_values = chi2(X_cat, y_binary)

print("=== Chi-square scores: scratch vs. sklearn ===")
df_chi2 = pd.DataFrame({
    'feature':        cat_names,
    'chi2_scratch':   chi2_scores_scratch,
    'chi2_sklearn':   chi2_scores_sklearn,
    'p_value':        p_values
}).sort_values('chi2_sklearn', ascending=False)
print(df_chi2.to_string(index=False))
print("\np < 0.05 → reject independence → feature carries information about the target.")

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(df_chi2['feature'], df_chi2['chi2_sklearn'], color='mediumseagreen')
ax.set_xlabel('Chi-square statistic')
ax.set_title('Chi-square Scores for Categorical Features')
plt.tight_layout()
plt.show()


## Step 5 — Redundancy Removal: Dropping Highly Correlated Pairs

### Intuition

After ranking by MI or Pearson r, the top-k list may include **redundant pairs**.
For example, `bedrooms` and `total_rooms` both rank high because both correlate with price.
But they also correlate *with each other* — feeding both into a model adds noise without adding
new information, and can cause multicollinearity in linear models.

**Strategy:** For every pair with $|r_{ij}| > \text{threshold}$ (e.g. 0.90),
keep the feature with higher target correlation and drop the other.


In [ ]:
np.random.seed(42)

def remove_correlated_features(X, feature_names, threshold=0.90):
    """
    Greedily drop the second feature in any pair whose |Pearson r| > threshold.
    Returns the indices of features to keep and the full correlation matrix.
    """
    corr_matrix = np.corrcoef(X.T)   # shape (p, p)
    to_drop = set()
    for i in range(len(feature_names)):
        for j in range(i + 1, len(feature_names)):
            # Only drop j if it hasn't already been marked for removal
            if abs(corr_matrix[i, j]) > threshold and j not in to_drop:
                to_drop.add(j)   # keep i (earlier in list), drop j (later)
    keep = [i for i in range(len(feature_names)) if i not in to_drop]
    return keep, corr_matrix

keep_idx, corr_mat = remove_correlated_features(X_vt, names_vt, threshold=0.90)
names_after_dedup = [names_vt[i] for i in keep_idx]
X_dedup = X_vt[:, keep_idx]

print(f"Features before dedup: {len(names_vt)}")
print(f"Features after  dedup: {len(names_after_dedup)}")
print(f"Dropped: {[n for n in names_vt if n not in names_after_dedup]}")

# ── Correlation heatmaps: before and after ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(pd.DataFrame(corr_mat, index=names_vt, columns=names_vt),
            ax=axes[0], cmap='coolwarm', center=0,
            vmin=-1, vmax=1, linewidths=0.4, annot=False)
axes[0].set_title('Correlation Matrix — Before Deduplication')

corr_after = np.corrcoef(X_dedup.T)
sns.heatmap(pd.DataFrame(corr_after, index=names_after_dedup, columns=names_after_dedup),
            ax=axes[1], cmap='coolwarm', center=0,
            vmin=-1, vmax=1, linewidths=0.4, annot=True, fmt='.2f')
axes[1].set_title('Correlation Matrix — After Deduplication')

plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

# Does removing redundant features hurt model performance?
ridge = Ridge(alpha=1.0)

r2_all_features = cross_val_score(ridge, X_vt,   y, cv=5, scoring='r2').mean()
r2_dedup        = cross_val_score(ridge, X_dedup, y, cv=5, scoring='r2').mean()

print(f"5-fold CV R²  —  all features after VT : {r2_all_features:.4f}")
print(f"5-fold CV R²  —  after dedup (thresh=0.9): {r2_dedup:.4f}")
print()
print("Removing redundant features rarely hurts and often helps (less multicollinearity).")


## Full Filter Pipeline

Putting it all together:

```
X_raw
  └─ Variance threshold (drop near-constants)
       └─ MI ranking (keep top-k)
            └─ Redundancy removal (drop correlated pairs)
                 └─ X_filtered  →  train model
```


In [ ]:
np.random.seed(42)

ridge = Ridge(alpha=1.0)

# Stage 0: baseline — all 15 original features
r2_stage0 = cross_val_score(ridge, X, y, cv=5, scoring='r2').mean()

# Stage 1: after variance threshold (using X_aug with near-constant features added)
mask_vt, _ = variance_threshold_scratch(X_aug, threshold=0.01)
X_s1       = X_aug[:, mask_vt]
r2_stage1  = cross_val_score(ridge, X_s1, y, cv=5, scoring='r2').mean()

# Stage 2: after MI ranking — keep top 10
mi_s2      = mutual_info_regression(X_s1, y, random_state=42)
top10_idx  = np.argsort(mi_s2)[::-1][:10]
X_s2       = X_s1[:, top10_idx]
r2_stage2  = cross_val_score(ridge, X_s2, y, cv=5, scoring='r2').mean()
names_s2   = [names_vt[i] for i in top10_idx]

# Stage 3: after redundancy removal
keep_s3, _  = remove_correlated_features(X_s2, names_s2, threshold=0.90)
X_s3        = X_s2[:, keep_s3]
r2_stage3   = cross_val_score(ridge, X_s3, y, cv=5, scoring='r2').mean()
names_s3    = [names_s2[i] for i in keep_s3]

print("=== Filter Pipeline: CV R² at each stage ===")
stages = [
    ('Stage 0: All features (15)',              X.shape[1],    r2_stage0),
    ('Stage 1: After variance threshold',        X_s1.shape[1], r2_stage1),
    ('Stage 2: After MI top-10',                 X_s2.shape[1], r2_stage2),
    ('Stage 3: After redundancy removal (0.9)',  X_s3.shape[1], r2_stage3),
]
for label, n_feats, r2 in stages:
    print(f"  {label:<45}  features={n_feats:2d}  R²={r2:.4f}")

print(f"\nFinal feature set: {names_s3}")

# ── Bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"Stage {i}\n({s[1]} feats)" for i, s in enumerate(stages)]
r2_vals = [s[2] for s in stages]
bars = ax.bar(labels, r2_vals, color=['#4c72b0','#55a868','#c44e52','#8172b3'])
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(min(r2_vals) * 0.95, max(r2_vals) * 1.02)
ax.set_ylabel('5-fold CV R²')
ax.set_title('Filter Pipeline: Model Performance at Each Stage')
plt.tight_layout()
plt.show()


## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** You have a binary feature that is 1 for only 3 out of 10,000 samples.
Why would a variance threshold filter likely remove it, and is that always the right decision?

<details><summary>Show answer</summary>

The variance of a Bernoulli variable with $p = 3/10{,}000 = 0.0003$ is $p(1-p) \approx 0.0003$,
which is below typical thresholds like 0.01.
The filter correctly removes it because:
1. Only 3 samples drive any split — the model would learn from noise.
2. In cross-validation some folds may contain zero positive examples.

However, in rare-event detection (fraud, medical diagnosis), a near-zero-prevalence feature
might be the most important signal (e.g., a specific genetic marker).
In those cases, **lower the threshold or skip variance filtering for known important features**.

</details>

---

**Q2.** Two features A and B both have |Pearson r| = 0.6 with the target, and
$r_{AB}$ = 0.95 with each other. Your redundancy removal drops B. Could dropping B ever
hurt model performance?

<details><summary>Show answer</summary>

Rarely, but yes. If A and B each capture a *slightly different* aspect of the target
(e.g., they are 95% correlated but B has 5% unique information), then dropping B loses
that 5%. In practice the loss is tiny, and keeping both causes multicollinearity in linear
models (unstable coefficients, inflated variance). The trade-off almost always favours dropping.

A safer approach: instead of dropping, apply **PCA** to the correlated pair and keep the first
principal component — this retains 100% of their combined variance.

</details>

---

**Q3.** Why can MI give a non-zero score even if x and y are linearly uncorrelated
(Pearson r ≈ 0)?

<details><summary>Show answer</summary>

Pearson r captures only *linear* co-movement. For a U-shaped relationship like $y = x^2$,
positive and negative x-values both produce high y-values — the positive and negative
deviations from the mean cancel out in the covariance formula, giving r ≈ 0.

MI does not care about *direction* of dependence. It asks: "Is the joint distribution
$p(x, y)$ different from the product of marginals $p(x) \cdot p(y)$?" For $y = x^2$,
the joint distribution clearly clusters along a parabola — very far from independence —
so MI is large.

</details>


## Key Takeaways

| Step | Method | When to use |
|---|---|---|
| **First** | Variance threshold | Always — instant removal of constants |
| **Second** | Mutual Information | Default ranking; handles non-linear relationships |
| **Second (alt)** | Pearson r | When you know relationships are linear and want speed |
| **Second (categoricals)** | Chi-square | When features and/or target are categorical |
| **Third** | Redundancy removal | After ranking — prevent multicollinearity |

**Bottom line for production:**
- Run the full filter pipeline to reduce from $p = 1000$ to $p \approx 30$–$50$ features.
- Then (optionally) run wrapper or embedded methods on the reduced set.
- Filter methods cannot detect feature *interactions* — e.g., `bedrooms × review_score`.
  For interactions, you need wrapper methods (Notebook 8) or tree-based embedded methods (Notebook 9).

**Next:** Notebook 8 — Wrapper Methods (RFE, Sequential Forward/Backward Selection).
